In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.tools import tool
import os
import pandas as pd 
import io 
import requests 
from langgraph.prebuilt import create_react_agent



load_dotenv()

api_key = os.getenv("OLLAMA_API_KEY")



In [28]:
!pip uninstall langchain langchain_openai langchain_ollama -y

Found existing installation: langchain 1.3.14
Uninstalling langchain-1.3.14:
  Successfully uninstalled langchain-1.3.14
Found existing installation: langchain-openai 1.4.1
Uninstalling langchain-openai-1.4.1:
  Successfully uninstalled langchain-openai-1.4.1
Found existing installation: langchain-ollama 1.1.0
Uninstalling langchain-ollama-1.1.0:
  Successfully uninstalled langchain-ollama-1.1.0


In [2]:
headers = {
   "Authorization": f"Bearer {api_key}"
}
llm = ChatOllama(
   base_url="https://ollama.com", # 원격 서버 주소
   model="gemma4:31b-cloud",
   client_kwargs={"headers": headers},
   temperature=0.2,
   reasoning=True
)

In [3]:
@tool
def get_weather(city : str) -> str:
    """
    이 함수는 현재 도시의 날씨 정보를 리턴하는 함수입니다. 
    input : city (도시 이름은 영어로)  예) seoul
    return : 해당 도시의 날씨 정보 문자열 
    """
    return requests.get(f"https://wttr.in/{city}?format=j1").text

from datetime import date 
@tool
def get_today() -> str:
    """
    이 함수는 현재 날짜를 리턴합니다. 
    """
    return str(date.today())


In [23]:
@tool
def get_exchange(date_: str, past: int = 3) -> str:
    """
    날짜의 실제 환율 정보 조회하는 함수
    input : date_ : 예) 2026-07-23 past : 오늘 날짜의 경우 3, 과거 날짜면 0을 입력
    return : 해당 날짜의 환율 정보
    """
    url = "https://www.kebhana.com/cms/rate/wpfxd651_01i_01.do"

    payload = {
        "ajax" : "true",
        "curCd" : "",
        "tmpInqStrDt" : date_,
        "pbldDvCd" : f"{past}",
        "pbldSqn" : "",
        "hid_key_data" : "",
        "inqStrDt" : date_.replace('-', ''),
        "inqKindCd" : "1",
        "hid_enc_data" : "",
        "requestTarget" : "searchContentDiv",
    }

    return pd.read_html(io.StringIO(requests.post(url, data=payload).text))[0]

In [22]:
get_exchange("2026-07-22", 0)

통화       현찰                              송금           \
                 통화     사실 때            파실 때            보낼 때     받을 때   
                 통화       환율 Spread       환율 Spread     보낼 때     받을 때   
0            미국 USD  1504.88   1.75  1453.12   1.75  1493.40  1464.60   
1      일본 JPY (100)   922.44   1.75   890.72   1.75   915.46   897.70   
2            유로 EUR  1720.52   1.99  1653.38   1.99  1703.81  1670.09   
3            중국 CNY   229.23   5.00   207.41   5.00   220.50   216.14   
4            홍콩 HKD   192.34   1.97   184.92   1.97   190.51   186.75   
5            태국 THB    45.94   5.00    41.14   6.00    44.19    43.33   
6            대만 TWD    51.64  13.10    41.10  10.00    46.16    45.16   
7           필리핀 PHP    26.36  10.00    22.01   8.20    24.20    23.74   
8          싱가포르 SGD  1168.10   1.99  1122.52   1.99  1156.76  1133.86   
9            호주 AUD  1053.73   1.97  1013.03   1.97  1043.71  1023.05   
10    베트남 VND (100)     6.29  11.80     4.97  11.80     5.68     5.58   
11           영국 GBP  2016.15   1.97  1938.25   1.97  1996.97  1957.43   
12          캐나다 CAD  1070.31   1.97  1028.97   1.97  1060.13  1039.15   
13        말레이시아 MYR   388.65   7.40   335.11   7.40   365.49   358.27   
14          러시아 RUB    20.77   9.50    15.66  17.50    19.63    18.31   
15        남아공화국 ZAR    97.27   8.00    82.87   8.00    91.15    88.99   
16         노르웨이 NOK   159.16   3.30   145.30   5.70   155.62   152.54   
17         뉴질랜드 NZD   876.15   1.97   842.31   1.97   867.82   850.64   
18          덴마크 DKK   233.10   3.30   212.80   5.70   227.91   223.41   
19          멕시코 MXN    93.85  10.50    76.03  10.50    85.78    84.10   
20           몽골 MNT     0.00   0.00     0.00   0.00     0.41     0.41   
21          바레인 BHD  4172.93   6.40  3608.18   8.00  3961.14  3882.72   
22        방글라데시 BDT     0.00   0.00     0.00   0.00    12.12    11.84   
23          브라질 BRL   322.19  10.20   263.14  10.00   295.87   288.87   
24         브루나이 BND  1191.12   4.00  1076.60   6.00  1159.05  1131.57   
25      사우디아라비아 SAR   418.81   6.30   366.81   6.90   397.92   390.06   
26         스리랑카 LKR     0.00   0.00     0.00   0.00     4.45     4.35   
27          스웨덴 SEK   157.48   3.30   143.77   5.70   153.97   150.93   
28          스위스 CHF  1851.26   1.97  1779.74   1.97  1833.65  1797.35   
29    아랍에미리트공화국 AED   424.84   5.50   374.92   6.90   406.72   398.68   
30          알제리 DZD     0.00   0.00     0.00   0.00    11.23    10.97   
31           오만 OMR  4110.46   7.00  3572.66   7.00  3887.65  3795.47   
32          요르단 JOD  2271.69   8.90  1919.16   8.00  2111.07  2061.01   
33         이스라엘 ILS   531.71  10.00   444.71   8.00   489.18   477.58   
34          이집트 EGP     0.00   0.00     0.00   0.00    29.17    28.49   
35           인도 INR     0.00   0.00     0.00   0.00    15.49    15.15   
36  인도네시아 IDR (100)     9.11  10.00     7.47  10.00     8.37     8.21   
37           체코 CZK    75.74   8.50    63.53   9.00    70.57    69.05   
38           칠레 CLP     1.73  10.00     1.43  10.00     1.59     1.57   
39        카자흐스탄 KZT     0.00   0.00     0.00   0.00     3.19     3.13   
40          카타르 QAR     0.00   0.00     0.00   0.00   410.78   401.04   
41           케냐 KES     0.00   0.00     0.00   0.00    11.55    11.29   
42         콜롬비아 COP     0.00   0.00     0.00   0.00     0.46     0.46   
43         쿠웨이트 KWD  5079.44   6.50  4387.88   8.00  4817.12  4721.74   
44         탄자니아 TZS     0.00   0.00     0.00   0.00     0.56     0.56   
45         튀르키예 TRY     0.00   0.00     0.00   0.00    31.67    30.99   
46         파키스탄 PKR     0.00   0.00     0.00   0.00     5.38     5.26   
47          폴란드 PLN   420.93   8.00   358.57   8.00   394.03   385.47   
48          헝가리 HUF     5.07   9.30     4.27   8.00     4.69     4.59   
49           네팔 NPR     0.00   0.00     0.00   0.00     9.68     9.46   
50          마카오 MOP     0.00   0.00     0.00   0.00   185.19   180.81   
51         캄보디아 KHR     0.00   0.00     0.00   0.00     0.37     0.37   
52

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=[get_weather,get_today,get_exchange],
    prompt="""
    규칙 :
    - 사용자 질문에 대답하기 전에 먼저 오늘 날짜를 확인한다. 
    - 모르는 내용은 모른다고 말할 것.
    """
)


/tmp/ipykernel_37039/533148118.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [25]:
rt = agent.invoke(
    {
        "messages" : [
            {
                "role" : 'user',
                'content' : "일본 환율 1달간의 추이 알려줘"
            }
        ]
    }
)


In [26]:
print(rt['messages'][-1].content)

최근 1달간의 일본 엔화(JPY 100) 환율 추이는 다음과 같습니다. (매매 기준율 기준)

*   **2026년 6월 24일:** 954.25원
*   **2026년 7월 7일:** 935.50원
*   **2026년 7월 24일 (오늘):** 893.87원

**분석:**
지난 한 달 동안 엔화 환율은 **지속적인 하락세**를 보이고 있습니다. 6월 말 950원대에서 시작하여 현재는 890원대까지 떨어지며 엔저 현상이 심화되는 추세입니다.


In [31]:
import pandas as pd

data_dir = "./data/"
dirs = ["상반기 주유소 판매가격", "하반기 주유소 판매가격"]

df = pd.concat(pd.read_csv(data_dir + dir + '.csv', encoding='euc-kr') for dir in dirs)

In [45]:
df.번호.unique()

<ArrowStringArray>
['A0006039', 'A0000525', 'A0001219', 'A0009061', 'A0001217', 'A0001213',
 'A0009838', 'A0010235', 'A0009710', 'A0010207',
 ...
 'A0004216', 'A0010071', 'A0032696', 'A0010081', 'A0009138', 'A0009220',
 'A0009197', 'A0009180', 'A0032659', 'A0033424']
Length: 512, dtype: str

In [50]:
len([len(df[df.번호 == x]) == 365 for x in df.번호.unique()])

512

In [57]:
강남 = df[df.지역 == "서울 강남구"].copy()
동작 = df[df.지역 == "서울 동작구"].copy()


In [61]:
강남_주유소_평균 = 강남.groupby(['번호'])['휘발유'].mean()
동작_주유소_평균 = 동작.groupby(['번호'])['휘발유'].mean()


In [60]:
print(강남_주유소_평균.shape)
print(동작_주유소_평균.shape)

(41,)
(10,)


In [63]:
from scipy import stats

In [64]:
stats.ttest_ind(강남_주유소_평균, 동작_주유소_평균, equal_var=False)

TtestResult(statistic=np.float64(6.50012362800111), pvalue=np.float64(4.0869804588151784e-08), df=np.float64(48.58979964464152))